# 10_transfer_learning.py
**Transfer Learning**

Configure frozen, partial, or full fine-tuning for downstream toxicity tasks.

This notebook is the one-to-one notebook version of `scripts/10_transfer_learning.py`. The implementation below is copied from that script so the Python and notebook workflows stay aligned.

## 1. Project setup
This cell locates the repository root and makes `scripts/core` imports available.

In [ ]:
from pathlib import Path
import os
import sys

# Make the notebook runnable whether Jupyter starts in the project root
# or directly inside the notebooks/ folder.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


## 2. Implementation

Every function below is copied verbatim from `scripts/10_transfer_learning`, in source order, kept in sync by `python scripts/sync_notebooks.py` (and checked by `11_validate_project.py`). Edit the `.py` file, then re-run the sync script -- never hand-edit these cells.

In [ ]:
"""STEP 10 -- Transfer-learning helper: reuses the step-03 foundation model's
pretrained embedding as a starting point for a small task head predicting a
real downstream endpoint (MEA, hERG, DILI, or another registered endpoint
from step 09), instead of training that task from scratch.

This file intentionally does not invent endpoint labels. It provides the three
fine-tuning strategies and the low-data experiment matrix. Once a downstream
dataset contains DTXSID/SMILES plus a real label, connect it here.
"""
import argparse
import torch.nn as nn

FRACTIONS = [0.10, 0.25, 0.50, 1.00]
STRATEGIES = ["scratch", "head_only", "partial", "full"]


#### `TaskHead`

Small MLP bolted onto the pretrained model's shared embedding to predict

In [ ]:
class TaskHead(nn.Module):
    """Small MLP bolted onto the pretrained model's shared embedding to predict
    one downstream endpoint (default: a single scalar/logit output)."""

    def __init__(self, latent_dim=256, output_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(128, output_dim),
        )

    def forward(self, embedding):
        return self.net(embedding)


#### `configure_finetuning`

Freeze/unfreeze `model`'s parameters in place according to `strategy`:

In [ ]:
def configure_finetuning(model, task_head, strategy="head_only", last_n_layers=2):
    """Freeze/unfreeze `model`'s parameters in place according to `strategy`:
    head_only trains just the new TaskHead; partial also unfreezes the last
    `last_n_layers` transformer layers + the fusion block; full unfreezes
    everything. The task head's own parameters are always trainable."""
    for parameter in model.parameters():
        parameter.requires_grad = False

    if strategy == "partial":
        for layer in model.transformer.layers[-last_n_layers:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True
        for parameter in model.fusion.parameters():
            parameter.requires_grad = True
    elif strategy == "full":
        for parameter in model.parameters():
            parameter.requires_grad = True
    elif strategy != "head_only":
        raise ValueError("strategy must be head_only, partial, or full")

    for parameter in task_head.parameters():
        parameter.requires_grad = True


#### `main`

CLI entry point: print the chosen strategy and, with --show-plan, the

In [ ]:
def main():
    """CLI entry point: print the chosen strategy and, with --show-plan, the
    full low-data experiment matrix (see module docstring)."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--strategy", choices=["head_only", "partial", "full"], default="head_only")
    parser.add_argument("--show-plan", action="store_true")
    args = parser.parse_args()
    print("Selected fine-tuning strategy:", args.strategy)
    if args.show_plan:
        print("\nRecommended low-data comparison:")
        for fraction in FRACTIONS:
            for strategy in STRATEGIES:
                print(f"fraction={fraction:>4.0%}  strategy={strategy}")
    print("\nNo synthetic labels are generated. Use real MEA/hERG/DILI/other endpoint labels.")


In [ ]:
# Set RUN_STEP=True when you are ready to execute this workflow.
# The notebook defaults to False so "Run All" is safe and does not accidentally
# download large files, start a long training job, or overwrite project outputs.
RUN_STEP = False

# Command-line arguments used when RUN_STEP=True.
RUN_ARGS = ["--show-plan"]

if RUN_STEP:
    old = sys.argv[:]
    try:
        sys.argv = ['10_transfer_learning.py'] + RUN_ARGS
        try:
            main()
        except SystemExit as exc:
            # main() uses SystemExit(0) as a CLI success signal (e.g. --check).
            # A terminal treats that as silent success; Jupyter displays *any*
            # SystemExit as an error-looking traceback, so only re-raise on an
            # actual failure (nonzero/non-None exit code).
            if exc.code not in (0, None):
                raise
    finally:
        sys.argv = old
else:
    print('Implementation loaded successfully.')
    print("Set RUN_STEP = True in this cell to execute: 10_transfer_learning.py")
    print('RUN_ARGS =', RUN_ARGS)
